# Question 3: Adversarial Search — Tic-Tac-Toe with Minimax

**Course:** Artificial Intelligence (ARI711S)  
**Assignment:** 1  
**Group:** Ninjas 2026

---

## Overview

We implement an AI agent that plays **Tic-Tac-Toe optimally** using the **Minimax algorithm**. The AI is unbeatable — it will always either win or draw, never lose.

### Problem Framing (Two-Player Adversarial Game)

| Element | Description |
|---------|-------------|
| **States** | 3×3 game boards |
| **Actions** | Empty positions `(i, j)` where a player can move |
| **Players** | X (maximiser) and O (minimiser), alternating turns |
| **Terminal states** | Board full, or one player has three in a row |
| **Utility** | +1 (X wins), −1 (O wins), 0 (tie) |

### Minimax Logic

- **X (MAX player):** Chooses the action that **maximises** the utility.
- **O (MIN player):** Chooses the action that **minimises** the utility.
- The algorithm explores all possible future game states recursively and backs up the optimal value at each node.

---

## 1. Imports and Constants

In [ ]:
import copy
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# Player tokens
X    = "X"
O    = "O"
EMPTY = None

def empty_board():
    """Return a fresh 3×3 board with all cells empty."""
    return [[EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY],
            [EMPTY, EMPTY, EMPTY]]

---
## 2. Core Game Logic Functions

### 2.1 `player(board)` — Determine Whose Turn It Is

In the initial state, **X moves first**. We count filled cells: if the counts of X and O are equal, it is X's turn; otherwise it is O's turn.

In [ ]:
def player(board):
    """
    Return which player's turn it is: X or O.

    X always goes first on an empty board. After that, turns alternate.
    If the board is terminal the return value is irrelevant.

    Parameters:
        board (list[list]): 3x3 board, cells are X, O, or None.

    Returns:
        str: 'X' or 'O'
    """
    x_count = sum(row.count(X) for row in board)
    o_count = sum(row.count(O) for row in board)

    # X has placed the same number of pieces as O → it is X's turn
    # X has placed one more piece than O → it is O's turn
    return X if x_count == o_count else O

### 2.2 `actions(board)` — List All Legal Moves

Any empty cell `(i, j)` is a valid action.

In [ ]:
def actions(board):
    """
    Return the set of all legal moves on the board.

    Each move is represented as a tuple (i, j) where:
        i = row index (0, 1, or 2)
        j = column index (0, 1, or 2)

    Parameters:
        board (list[list]): Current game board.

    Returns:
        set[tuple]: Set of (i, j) tuples for all empty cells.
    """
    return {
        (i, j)
        for i in range(3)
        for j in range(3)
        if board[i][j] == EMPTY
    }

### 2.3 `result(board, action)` — Apply a Move

Returns a **new board** with the action applied, without modifying the original. This is essential for Minimax, which must explore many branches simultaneously.

In [ ]:
def result(board, action):
    """
    Return the board that results from the current player making 'action'.

    The original board is NOT modified (deep copy is made first).

    Parameters:
        board  (list[list]): Current game board.
        action (tuple)     : (i, j) — cell where the current player moves.

    Returns:
        list[list]: New board state after the move.

    Raises:
        ValueError: If the action is not valid (cell is already occupied).
    """
    i, j = action

    if board[i][j] != EMPTY:
        raise ValueError(f"Action {action} is not valid: cell is already occupied by '{board[i][j]}'.")

    # Deep copy so the original board remains unmodified
    new_board = copy.deepcopy(board)
    new_board[i][j] = player(board)  # Current player makes the move
    return new_board

### 2.4 `winner(board)` — Check for a Winner

Checks all 8 winning lines: 3 rows, 3 columns, and 2 diagonals.

In [ ]:
def winner(board):
    """
    Return the winner of the game, or None if there is no winner yet.

    A player wins by filling an entire row, column, or diagonal.

    Parameters:
        board (list[list]): Current game board.

    Returns:
        str or None: 'X', 'O', or None.
    """
    # All 8 lines to check: 3 rows + 3 columns + 2 diagonals
    lines = [
        # Rows
        [(0,0), (0,1), (0,2)],
        [(1,0), (1,1), (1,2)],
        [(2,0), (2,1), (2,2)],
        # Columns
        [(0,0), (1,0), (2,0)],
        [(0,1), (1,1), (2,1)],
        [(0,2), (1,2), (2,2)],
        # Diagonals
        [(0,0), (1,1), (2,2)],
        [(0,2), (1,1), (2,0)],
    ]

    for line in lines:
        values = [board[r][c] for r, c in line]
        if values[0] is not None and values[0] == values[1] == values[2]:
            return values[0]  # Returns 'X' or 'O'

    return None  # No winner

### 2.5 `terminal(board)` — Check if the Game Is Over

The game ends when there is a winner **or** the board is completely filled (tie).

In [ ]:
def terminal(board):
    """
    Return True if the game is over, False if it is still in progress.

    The game is over when:
        - A player has won (three in a row/column/diagonal), OR
        - All 9 cells are filled (tie).

    Parameters:
        board (list[list]): Current game board.

    Returns:
        bool
    """
    # Game over if someone has won
    if winner(board) is not None:
        return True

    # Game over if no empty cells remain (tie)
    for row in board:
        for cell in row:
            if cell == EMPTY:
                return False  # Still at least one empty cell

    return True  # All cells filled, no winner → tie

### 2.6 `utility(board)` — Assign a Score to a Terminal Board

Utility is only called when `terminal(board)` is True.

In [ ]:
def utility(board):
    """
    Return the utility value of a terminal board state.

        +1 if X has won
        -1 if O has won
         0 for a tie

    This function should only be called when terminal(board) is True.

    Parameters:
        board (list[list]): A terminal game board.

    Returns:
        int: 1, -1, or 0.
    """
    w = winner(board)
    if w == X:
        return 1
    elif w == O:
        return -1
    else:
        return 0  # Tie

---
## 3. Minimax Algorithm

### How Minimax Works

Minimax recursively explores the full game tree:
- **X (MAX):** At X's turn, choose the action that leads to the highest utility.
- **O (MIN):** At O's turn, choose the action that leads to the lowest utility.
- At **terminal** nodes, return the utility directly.

We implement two helper functions `max_value` and `min_value` that back up scores through the tree, and the main `minimax` function that returns the best action.

### Alpha-Beta Pruning (Optimisation)

We also implement **alpha-beta pruning**, which skips branches that cannot possibly influence the final decision. This reduces the number of nodes explored without changing the result — the AI is still perfectly optimal.

In [ ]:
def max_value(board, alpha, beta):
    """
    Return the maximum achievable utility from this board state.
    Called when it is X's turn (the maximising player).

    Uses alpha-beta pruning to skip branches that cannot improve the result.

    Parameters:
        board (list[list]): Current board.
        alpha (float)     : Best value MAX can guarantee so far.
        beta  (float)     : Best value MIN can guarantee so far.

    Returns:
        int: The utility value backed up from this node.
    """
    if terminal(board):
        return utility(board)

    v = -math.inf
    for action in actions(board):
        v = max(v, min_value(result(board, action), alpha, beta))
        alpha = max(alpha, v)
        if alpha >= beta:  # Beta cut-off: MIN would never allow this branch
            break
    return v


def min_value(board, alpha, beta):
    """
    Return the minimum achievable utility from this board state.
    Called when it is O's turn (the minimising player).

    Parameters:
        board (list[list]): Current board.
        alpha (float)     : Best value MAX can guarantee so far.
        beta  (float)     : Best value MIN can guarantee so far.

    Returns:
        int: The utility value backed up from this node.
    """
    if terminal(board):
        return utility(board)

    v = math.inf
    for action in actions(board):
        v = min(v, max_value(result(board, action), alpha, beta))
        beta = min(beta, v)
        if alpha >= beta:  # Alpha cut-off: MAX would never allow this branch
            break
    return v


def minimax(board):
    """
    Return the optimal action for the current player on the given board.

    - If it is X's turn: choose the action with the highest min_value.
    - If it is O's turn: choose the action with the lowest max_value.
    - If the board is terminal: return None.

    Parameters:
        board (list[list]): Current game board.

    Returns:
        tuple or None: (i, j) optimal action, or None if game is over.
    """
    if terminal(board):
        return None

    current = player(board)
    best_action = None

    if current == X:
        # X is the MAX player — find action with highest value
        best_score = -math.inf
        for action in actions(board):
            score = min_value(result(board, action), -math.inf, math.inf)
            if score > best_score:
                best_score = score
                best_action = action
    else:
        # O is the MIN player — find action with lowest value
        best_score = math.inf
        for action in actions(board):
            score = max_value(result(board, action), -math.inf, math.inf)
            if score < best_score:
                best_score = score
                best_action = action

    return best_action

---
## 4. Board Display Utilities

Helper functions to print and visualise board states.

In [ ]:
def print_board(board, label=""):
    """Print the board to the console in a readable format."""
    symbols = {X: 'X', O: 'O', EMPTY: '.'}
    if label:
        print(f"\n{label}")
    print("  0 1 2")
    for i, row in enumerate(board):
        print(f"{i} " + " ".join(symbols[cell] for cell in row))


def plot_board(board, title="", highlight=None, ax=None):
    """
    Render a Tic-Tac-Toe board using matplotlib.

    Parameters:
        board     (list[list]) : Game board.
        title     (str)        : Plot title.
        highlight (tuple)      : (i, j) cell to highlight in yellow.
        ax        (Axes)       : Existing axes (if embedding in a subplot).
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(3.5, 3.5))

    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3)
    ax.set_aspect('equal')
    ax.axis('off')

    # Grid lines
    for x in [1, 2]:
        ax.axvline(x, color='#333333', linewidth=2)
    for y in [1, 2]:
        ax.axhline(y, color='#333333', linewidth=2)

    # Background
    ax.set_facecolor('#f7f7f7')

    # Highlight a cell
    if highlight:
        hi_r, hi_c = highlight
        rect = plt.Rectangle(
            (hi_c, 2 - hi_r), 1, 1,
            color='#fffacd', zorder=0
        )
        ax.add_patch(rect)

    # Draw X and O symbols
    for i in range(3):
        for j in range(3):
            cell = board[i][j]
            cx = j + 0.5
            cy = (2 - i) + 0.5
            if cell == X:
                # Draw X as two crossing lines
                ax.plot([cx - 0.3, cx + 0.3], [cy - 0.3, cy + 0.3],
                        color='#e74c3c', linewidth=4, solid_capstyle='round')
                ax.plot([cx + 0.3, cx - 0.3], [cy - 0.3, cy + 0.3],
                        color='#e74c3c', linewidth=4, solid_capstyle='round')
            elif cell == O:
                # Draw O as a circle
                circle = plt.Circle((cx, cy), 0.32,
                                    color='#2980b9', fill=False, linewidth=4)
                ax.add_patch(circle)

    if title:
        ax.set_title(title, fontsize=10, pad=6)

    if standalone:
        plt.tight_layout()
        plt.show()

---
## 5. Test Cases

We test each helper function before running full games.

### 5.1 — `player()` and `actions()`

In [ ]:
print("=" * 45)
print("TEST: player() and actions()")
print("=" * 45)

# Empty board: X goes first
b0 = empty_board()
print(f"Empty board → player: {player(b0)} (expected: X)")
print(f"Empty board → actions count: {len(actions(b0))} (expected: 9)")

# After X plays centre
b1 = result(b0, (1, 1))
print_board(b1, "After X plays (1,1):")
print(f"  player: {player(b1)} (expected: O)")
print(f"  actions count: {len(actions(b1))} (expected: 8)")

# After O plays top-left
b2 = result(b1, (0, 0))
print_board(b2, "After O plays (0,0):")
print(f"  player: {player(b2)} (expected: X)")

### 5.2 — `winner()` and `terminal()`

In [ ]:
print("=" * 45)
print("TEST: winner() and terminal()")
print("=" * 45)

# X wins via top row
b_x_wins = [
    [X, X, X],
    [O, O, None],
    [None, None, None]
]
print_board(b_x_wins, "X wins via top row:")
print(f"  winner: {winner(b_x_wins)} (expected: X)")
print(f"  terminal: {terminal(b_x_wins)} (expected: True)")

# O wins via left column
b_o_wins = [
    [O, X, X],
    [O, X, None],
    [O, None, None]
]
print_board(b_o_wins, "O wins via left column:")
print(f"  winner: {winner(b_o_wins)} (expected: O)")
print(f"  terminal: {terminal(b_o_wins)} (expected: True)")

# X wins via main diagonal
b_diag = [
    [X, O, O],
    [None, X, O],
    [None, None, X]
]
print_board(b_diag, "X wins via diagonal:")
print(f"  winner: {winner(b_diag)} (expected: X)")

# Tie
b_tie = [
    [X, O, X],
    [X, X, O],
    [O, X, O]
]
print_board(b_tie, "Tie game:")
print(f"  winner: {winner(b_tie)} (expected: None)")
print(f"  terminal: {terminal(b_tie)} (expected: True)")
print(f"  utility: {utility(b_tie)} (expected: 0)")

### 5.3 — `result()` — Immutability Check

In [ ]:
print("=" * 45)
print("TEST: result() — immutability")
print("=" * 45)

original = empty_board()
new = result(original, (0, 0))

print(f"Original board[0][0]: {original[0][0]} (expected: None — unchanged)")
print(f"New board[0][0]:      {new[0][0]}      (expected: X — move applied)")

# Invalid action should raise an error
try:
    result(new, (0, 0))  # (0,0) is already occupied
    print("ERROR: Should have raised ValueError")
except ValueError as e:
    print(f"Correctly raised ValueError: {e}")

### 5.4 — `minimax()` — Optimal Move Verification

In [ ]:
print("=" * 45)
print("TEST: minimax() — optimal decisions")
print("=" * 45)

# Scenario 1: X must win on next move
# X . X        →   X should play (0,1) to complete the top row
# O O .
# . . .
board_win_now = [
    [X,    None, X],
    [O,    O,    None],
    [None, None, None]
]
move = minimax(board_win_now)
print_board(board_win_now, "X to move — must win immediately:")
print(f"  minimax move: {move} (expected: (0,1))")

# Scenario 2: O must block or lose
# X X .
# O . .
# . . .
board_block = [
    [X,    X,    None],
    [O,    None, None],
    [None, None, None]
]
move2 = minimax(board_block)
print_board(board_block, "O to move — must block X at (0,2):")
print(f"  minimax move: {move2} (expected: (0,2))")

# Scenario 3: Empty board — optimal first move
move3 = minimax(empty_board())
print(f"\nOptimal first move on empty board: {move3}")
print("  (Centre (1,1) or any corner are all optimal — any is acceptable)")

---
## 6. Full Game Simulations

We simulate three complete games to demonstrate the AI's behaviour:
1. **AI (X) vs AI (O)** — both play optimally → always a tie
2. **AI (X) vs Human (O plays badly)** — X wins
3. **Human (X plays badly) vs AI (O)** — O wins

In [ ]:
def simulate_game(x_strategy, o_strategy, game_label="Game"):
    """
    Simulate a complete Tic-Tac-Toe game.

    Parameters:
        x_strategy (callable): Function(board) → (i, j) for X's moves.
        o_strategy (callable): Function(board) → (i, j) for O's moves.
        game_label (str)     : Label for display.

    Returns:
        list[list]: Board states at each step (for visualisation).
    """
    board = empty_board()
    history = [copy.deepcopy(board)]
    moves_taken = []

    print(f"\n{'='*50}")
    print(f"{game_label}")
    print(f"{'='*50}")
    print_board(board, "Initial board:")

    while not terminal(board):
        current = player(board)
        if current == X:
            action = x_strategy(board)
        else:
            action = o_strategy(board)

        board = result(board, action)
        moves_taken.append((current, action))
        history.append(copy.deepcopy(board))
        print_board(board, f"{current} plays {action}:")

    w = winner(board)
    if w:
        print(f"\nResult: {w} WINS!")
    else:
        print("\nResult: TIE!")

    return history, moves_taken

In [ ]:
# -------------------------------------------------------
# Game 1: Minimax X vs Minimax O (both optimal)
# -------------------------------------------------------
history1, moves1 = simulate_game(
    x_strategy=minimax,
    o_strategy=minimax,
    game_label="Game 1: AI (X) vs AI (O) — Both Optimal"
)

In [ ]:
# -------------------------------------------------------
# Game 2: Minimax X vs a poor O strategy
# (O always picks the first available cell — top-left scan)
# -------------------------------------------------------
def naive_strategy(board):
    """Always pick the first available empty cell (row-major order)."""
    for i in range(3):
        for j in range(3):
            if board[i][j] == EMPTY:
                return (i, j)

history2, moves2 = simulate_game(
    x_strategy=minimax,
    o_strategy=naive_strategy,
    game_label="Game 2: AI (X, Minimax) vs Human (O, Naive)"
)

In [ ]:
# -------------------------------------------------------
# Game 3: Naive X vs Minimax O
# -------------------------------------------------------
history3, moves3 = simulate_game(
    x_strategy=naive_strategy,
    o_strategy=minimax,
    game_label="Game 3: Human (X, Naive) vs AI (O, Minimax)"
)

---
## 7. Visual Game Progression

We display the move-by-move board states for all three games.

In [ ]:
def plot_game_progression(history, moves_taken, game_title):
    """
    Plot all board states of a game in a single figure row.

    Parameters:
        history     (list): Sequence of board states.
        moves_taken (list): List of (player, action) tuples.
        game_title  (str) : Overall figure title.
    """
    n = len(history)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))

    if n == 1:
        axes = [axes]

    for idx, (ax, board) in enumerate(zip(axes, history)):
        highlight = moves_taken[idx - 1][1] if idx > 0 else None
        if idx == 0:
            step_label = "Start"
        else:
            p, act = moves_taken[idx - 1]
            step_label = f"Move {idx}: {p}→{act}"

        # Mark the final state
        if idx == len(history) - 1:
            w = winner(board)
            if w:
                step_label += f"\n✓ {w} WINS"
            else:
                step_label += "\n= TIE"

        plot_board(board, title=step_label, highlight=highlight, ax=ax)

    plt.suptitle(game_title, fontsize=12, y=1.04)
    plt.tight_layout()
    fname = game_title.replace(" ", "_").replace(":", "")[:40] + ".png"
    plt.savefig(fname, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")


plot_game_progression(history1, moves1, "Game 1: AI(X) vs AI(O) — Tie Expected")
plot_game_progression(history2, moves2, "Game 2: AI(X) vs Naive(O)")
plot_game_progression(history3, moves3, "Game 3: Naive(X) vs AI(O)")

---
## 8. Interactive Text-Based Game (Human vs AI)

Run the cell below to play against the Minimax AI in the terminal. The AI plays as **O** and you play as **X**. Enter your move as `row col` (e.g. `1 1` for the centre).

In [ ]:
def play_human_vs_ai():
    """
    Interactive Human (X) vs AI (O) game in the notebook console.
    Human enters moves as: row col  (e.g. '1 1' for centre)
    """
    board = empty_board()
    print("\nTic-Tac-Toe: You are X, AI is O")
    print("Enter moves as: row col  (0-indexed, e.g. '1 1' for centre)\n")
    print_board(board)

    while not terminal(board):
        current = player(board)

        if current == X:
            # Human's turn
            valid = False
            while not valid:
                try:
                    inp = input("\nYour move (row col): ").strip().split()
                    i, j = int(inp[0]), int(inp[1])
                    if (i, j) not in actions(board):
                        print("  That cell is already taken. Try again.")
                    else:
                        valid = True
                except (ValueError, IndexError):
                    print("  Invalid input. Enter two numbers, e.g. '0 2'")
            board = result(board, (i, j))
        else:
            # AI's turn
            print("\nAI is thinking...")
            action = minimax(board)
            board = result(board, action)
            print(f"AI plays: {action}")

        print_board(board)

    w = winner(board)
    if w == X:
        print("\nYou win! (But did the AI make a mistake? It should never lose…)")
    elif w == O:
        print("\nAI wins! Try again.")
    else:
        print("\nIt's a tie! Well played.")


# Uncomment the line below to play interactively:
# play_human_vs_ai()

---
## 9. Summary and Analysis

### Function Specification Summary

| Function | Input | Output | Notes |
|----------|-------|--------|-------|
| `player(board)` | Board | `'X'` or `'O'` | Counts pieces; X leads on even total |
| `actions(board)` | Board | `set` of `(i,j)` | All empty cells |
| `result(board, action)` | Board, action | New board | Deep copy; raises `ValueError` if cell occupied |
| `winner(board)` | Board | `'X'`, `'O'`, or `None` | Checks 8 lines |
| `terminal(board)` | Board | `bool` | True if winner exists or board full |
| `utility(board)` | Terminal board | `1`, `-1`, or `0` | Only valid on terminal states |
| `minimax(board)` | Board | `(i,j)` or `None` | Uses alpha-beta pruning for efficiency |

### Why the AI is Unbeatable

Minimax exhaustively evaluates **every possible future game state** and always selects the action that leads to the best guaranteed outcome:
- If the AI can win, it will.
- If it cannot win, it will draw.
- It never makes a move that lets the opponent win when a draw was available.

This is possible in Tic-Tac-Toe because the game tree is small enough (at most 9! = 362,880 leaf nodes) that full exploration is feasible, even without alpha-beta pruning.

### Alpha-Beta Pruning

Alpha-beta pruning reduces the effective branching factor without changing the result. The two parameters:
- **α (alpha):** The best score MAX has found so far on the path to the root.
- **β (beta):** The best score MIN has found so far on the path to the root.

A branch is pruned when the current node's value cannot possibly improve on what the opponent would already avoid: `α ≥ β`. This can reduce the nodes explored from O(b^d) to O(b^(d/2)) in the best case.

### Game 1 Result: Tie (as expected)

When both players play optimally, Tic-Tac-Toe always ends in a draw. This is a known game-theoretic result — the game is a **draw under optimal play** for both sides.